# 01 · Exploración del feed de USGS

## ¿Qué hacemos aquí y por qué?

Antes de construir el sistema (productor, Kafka, base de datos…) tenemos que **conocer los datos que vamos
a recibir**. Es como un cocinero que, antes de diseñar el menú, abre las cajas del proveedor para ver qué trae,
en qué estado llega y qué viene roto.

Respondemos a preguntas muy concretas, y cada respuesta se convertirá en una **regla del consumidor**:

| Pregunta | Regla que saldrá |
|---|---|
| ¿Qué campos trae cada terremoto? | Qué columnas tendrá nuestra base de datos |
| ¿Faltan valores? | Qué hacemos con los nulos |
| ¿Hay cosas que no son terremotos? | Filtro de `type` |
| ¿Hay valores raros (profundidad negativa…)? | Si se aceptan, se corrigen o van a cuarentena |
| ¿En qué unidades vienen las fechas? | Cómo convertirlas a UTC |

**Para la nota:** es la evidencia del criterio **Extracción** («carga + análisis de tipos» = sobresaliente).

**Datos:** una respuesta real del feed guardada el 28/10/2026 a las 11:36 UTC (187 terremotos).
⚠️ Es una foto de un momento: las cifras describen *esta* respuesta, no el feed en general.

**Cómo usarlo:** ejecuta las celdas de una en una (▶ o `Shift + Enter`) y lee el resultado debajo de cada una.

## 0. Cargar el archivo

**Idea:** el archivo es JSON (texto con llaves y corchetes). Python lo convierte en un **diccionario**
(`dict`), igual que un array asociativo de PHP: `feed["metadata"]` funciona como `$feed['metadata']`.

In [1]:
import json                 # librería estándar de Python para leer/escribir JSON
from pathlib import Path    # para construir rutas de archivos que funcionan en Windows y Linux

import pandas as pd         # pandas: tablas de datos (DataFrame), como una hoja de Excel programable

# Ruta al archivo. ".." = subir una carpeta (de analisis/ a la raíz del proyecto).
# El operador "/" de Path une trozos de ruta: ../docs/material-profesor/ejemplos/...
RUTA = Path("..") / "docs" / "material-profesor" / "ejemplos" / "feed-all_day-2026-10-28.geojson"

with open(RUTA, encoding="utf-8") as f:   # abre el archivo (y lo cierra solo al terminar el bloque)
    feed = json.load(f)                     # convierte el texto JSON en un diccionario de Python

print("Versión de pandas:", pd.__version__)
print("Claves de primer nivel:", list(feed.keys()))   # qué "cajones" tiene el diccionario
feed["metadata"]                                      # la última línea de una celda se muestra sola

Versión de pandas: 3.0.0
Claves de primer nivel: ['type', 'metadata', 'features', 'bbox']


{'generated': 1793187366000,
 'url': 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_day.geojson',
 'title': 'USGS All Earthquakes, Past Day',
 'status': 200,
 'api': '2.8.2',
 'count': 187}

**Qué deberías ver:** 4 claves (`type`, `metadata`, `features`, `bbox`) y en `metadata` el `count` = 187
(cuántos terremotos trae) y `generated` (cuándo se generó, en milisegundos).

## 1. ¿Cómo es UN terremoto?

**Idea:** antes de mirar 187, miramos 1 entero, tal cual llega. `features` es una **lista**;
`[0]` es el primer elemento (en Python se empieza a contar en 0).

In [2]:
print("Número de terremotos en la lista:", len(feed["features"]))   # len() = longitud de la lista
print("Lo que dice metadata.count:      ", feed["metadata"]["count"]) # deberían coincidir

feed["features"][0]   # el primer terremoto completo

Número de terremotos en la lista: 187
Lo que dice metadata.count:       187


{'type': 'Feature',
 'properties': {'mag': 0.75,
  'place': '5 km NW of The Geysers, CA',
  'time': 1793186692000,
  'updated': 1793186787595,
  'tz': None,
  'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/nc75458882',
  'detail': 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/nc75458882.geojson',
  'felt': None,
  'cdi': None,
  'mmi': None,
  'alert': None,
  'status': 'automatic',
  'tsunami': 0,
  'sig': 9,
  'net': 'nc',
  'code': '75458882',
  'ids': ',nc75458882,',
  'sources': ',nc,',
  'types': ',nearby-cities,origin,phase-data,',
  'nst': 8,
  'dmin': 0.007189,
  'rms': 0.03,
  'gap': 124,
  'magType': 'md',
  'type': 'earthquake',
  'title': 'M 0.8 - 5 km NW of The Geysers, CA'},
 'geometry': {'type': 'Point',
  'coordinates': [-122.795166015625, 38.8151664733887, 2.74000000953674]},
 'id': 'nc75458882'}

**Fíjate en:** `id` (fuera de `properties`), `geometry.coordinates` = `[longitud, latitud, profundidad]`,
y dentro de `properties`: `mag`, `time`, `updated`, `type`, `status`…

## 2. Convertir la lista en una tabla

**Idea:** con una lista de diccionarios anidados no se pueden hacer recuentos cómodos.
Queremos una **tabla**: una fila por terremoto, una columna por campo.

- `pd.json_normalize` "aplana": `{"properties": {"mag": 4}}` se convierte en una columna `properties.mag`.
- Las coordenadas llegan como lista de 3 números en una sola columna; las separamos en 3 columnas.
- Hay **dos campos llamados `type`** (el del Feature, siempre "Feature", y `properties.type`, que es
  "earthquake" o "explosion"). Si no los renombramos, chocan. *Esto lo descubrimos al ejecutar: es un
  hallazgo de la exploración.*

In [3]:
# 1) Aplanar: cada terremoto = una fila
df = pd.json_normalize(feed["features"])

# 2) Separar coordenadas: la columna "geometry.coordinates" tiene listas [lon, lat, prof].
#    .tolist() las saca como lista de listas y pd.DataFrame las convierte en 3 columnas con nombre.
coords = pd.DataFrame(df["geometry.coordinates"].tolist(), columns=["lon", "lat", "depth_km"])

# 3) Pegar las 3 columnas nuevas (axis=1 = pegar en horizontal) y quitar la columna original
df = pd.concat([df.drop(columns=["geometry.coordinates"]), coords], axis=1)

# 4) Renombrar los dos "type" ANTES de simplificar nombres, para que no choquen
df = df.rename(columns={"type": "feature_type", "geometry.type": "geom_type"})

# 5) Quitar el prefijo "properties." de los nombres de columna (properties.mag -> mag)
df.columns = [c.replace("properties.", "") for c in df.columns]

print("¿Nombres de columna únicos?", df.columns.is_unique)   # True = ningún nombre repetido
print("Tamaño (filas, columnas):", df.shape)
df.head()   # muestra las 5 primeras filas

¿Nombres de columna únicos? True
Tamaño (filas, columnas): (187, 32)


,feature_type,id,mag,place,time,updated,tz,url,detail,felt,...,dmin,rms,gap,magType,type,title,geom_type,lon,lat,depth_km
0,Feature,nc75458882,0.75,"5 km NW of The Geysers, CA",1793186692000,1793186787595,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/earthquakes/feed/v...,NaN,...,0.007189,0.0300,124.0,md,earthquake,"M 0.8 - 5 km NW of The Geysers, CA",Point,-122.795166,38.815166,2.7400
1,Feature,aka2026vizekq,1.30,"56 km E of McCarthy, Alaska",1793186283753,1793186382568,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/earthquakes/feed/v...,NaN,...,0.300000,0.5000,117.0,ml,earthquake,"M 1.3 - 56 km E of McCarthy, Alaska",Point,-141.881000,61.343000,5.0000
2,Feature,nn00926354,2.05,"12 km NNW of Silver Peak, Nevada",1793186144407,1793186299418,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/earthquakes/feed/v...,NaN,...,0.443000,0.3047,99.0,ml,earthquake,"M 2.1 - 12 km NNW of Silver Peak, Nevada",Point,-117.705600,37.847900,2.1122
3,Feature,aka2026viylbm,3.00,"25 km ESE of Kasilof, Alaska",1793184977538,1793185531613,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/earthquakes/feed/v...,1.0,...,0.500000,0.9000,79.0,ml,earthquake,"M 3.0 - 25 km ESE of Kasilof, Alaska",Point,-150.857000,60.243000,70.5000
4,Feature,aka2026viyhxm,2.20,"16 km SE of Denali National Park, Alaska",1793184730771,1793184861645,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/earthquakes/feed/v...,NaN,...,0.300000,0.8000,74.0,ml,earthquake,"M 2.2 - 16 km SE of Denali National Park, Alaska",Point,-151.477000,63.440000,5.0000


## 3. ¿Qué campos vienen rellenos y cuáles vacíos? → **Completitud**

**Idea:** es la primera de las 6 dimensiones de calidad de la sesión 7. Para cada columna contamos
cuántos valores hay y cuántos faltan (`null` en JSON = `NaN`/`None` en pandas).

**Por qué importa:** si un campo casi siempre viene vacío, no podemos basar un análisis en él.
Si un campo clave (`mag`) puede venir vacío, hay que decidir qué hacer (el enunciado: **no** poner 0).

In [4]:
resumen = pd.DataFrame({
    "tipo_pandas": df.dtypes.astype(str),              # tipo que pandas ha detectado en cada columna
    "informados":  df.notna().sum(),                    # notna() = True si hay valor; sum() cuenta los True
    "nulos":       df.isna().sum(),                     # isna() = True si falta el valor
    "%_informado": (df.notna().mean() * 100).round(1),  # mean() de True/False = proporción de True
}).sort_values("%_informado")                           # ordenamos: los más vacíos arriba

resumen

,tipo_pandas,informados,nulos,%_informado
tz,object,0,187,0.0
alert,str,6,181,3.2
mmi,float64,9,178,4.8
cdi,float64,16,171,8.6
felt,float64,16,171,8.6
feature_type,str,187,0,100.0
mag,float64,187,0,100.0
id,str,187,0,100.0
url,str,187,0,100.0
updated,int64,187,0,100.0


**Qué deberías ver:** arriba `tz` (0 %), `alert`, `mmi`, `cdi`, `felt` (casi vacíos: datos opcionales
que no necesitamos). Los campos que nos importan (`id`, `mag`, `time`, `updated`, `lat`, `lon`, `depth_km`) al 100 %.

## 4. ¿Se repite algún terremoto? → **Unicidad**

**Idea:** si el mismo `id` aparece dos veces en una respuesta, tendríamos duplicados *dentro* de ella.

**Por qué importa:** el problema gordo del proyecto son los duplicados **entre** respuestas (cada minuto
el feed vuelve a mandar los mismos 187). Aquí comprobamos que, al menos dentro de una, no los hay.

In [5]:
print("Filas:", len(df))
print("id distintos:", df["id"].nunique())   # nunique() = cuántos valores diferentes hay

Filas: 187
id distintos: 187


## 5. ¿Todo son terremotos? → filtro de `type`

**Idea:** USGS también registra explosiones (canteras, voladuras…). El enunciado manda quedarse solo con
`type == "earthquake"` y **guardar el motivo** de lo que se descarta.

In [6]:
df["type"].value_counts()   # value_counts() = cuántas veces aparece cada valor (como un GROUP BY + COUNT)

type
earthquake    185
explosion       2
Name: count, dtype: int64

In [7]:
# Filtro: filas cuyo type NO es "earthquake". df.loc[condición, columnas] = SELECT columnas WHERE condición
df.loc[df["type"] != "earthquake", ["id", "type", "mag", "depth_km", "place"]]

,id,type,mag,depth_km,place
114,uw714129552,explosion,0.40,-0.54,"9 km ENE of Hoodsport, Washington"
129,uw714129332,explosion,2.15,-0.62,"15 km S of Princeton, Canada"


## 6. Magnitud

**Idea:** rango de valores, cuántos superan 4 (el filtro de la comparación histórica) y qué escalas
(`magType`) aparecen.

**Por qué importa:** `ml`, `md`, `mb`, `mww` son métodos de medida distintos; mezclarlos en una media es
una limitación que hay que mencionar (dimensión **Consistencia**).

In [8]:
print(df["mag"].describe())                          # count, media, desviación, mínimo, cuartiles, máximo
print("\nTerremotos con mag >= 4:", (df["mag"] >= 4).sum())   # comparar da True/False; sum() cuenta los True
df["magType"].value_counts()

count    187.000000
mean       2.009223
std        1.401058
min        0.220000
25%        0.975000
50%        1.700000
75%        2.355000
max        6.000000
Name: mag, dtype: float64

Terremotos con mag >= 4: 28


magType
ml     108
md      51
mb      22
mww      6
Name: count, dtype: int64

## 7. Coordenadas y profundidad → **Validez**

**Idea:** comprobar que todo es un punto válido (latitud entre −90 y 90, longitud entre −180 y 180) y
mirar los valores raros de profundidad.

**Por qué importa:** una coordenada imposible iría a **cuarentena**. Una profundidad negativa NO es
imposible: USGS explica que un terremoto muy superficial puede salir negativo (ver memoria §4).

In [9]:
print(df["geom_type"].value_counts())   # ¿todos son "Point"?
print()

# Condición: latitud fuera de [-90, 90] O longitud fuera de [-180, 180]. "|" = OR; .abs() = valor absoluto
fuera_rango = df[(df["lat"].abs() > 90) | (df["lon"].abs() > 180)]
print("Coordenadas fuera de rango:", len(fuera_rango))

print(df["depth_km"].describe())

# Terremotos con profundidad negativa
df.loc[df["depth_km"] < 0, ["id", "type", "mag", "depth_km", "place"]]

geom_type
Point    187
Name: count, dtype: int64

Coordenadas fuera de rango: 0
count    187.000000
mean      29.716328
std       52.996813
min       -2.310000
25%        5.000000
50%       10.000000
75%       30.560000
max      509.065000
Name: depth_km, dtype: float64


,id,type,mag,depth_km,place
114,uw714129552,explosion,0.40,-0.54,"9 km ENE of Hoodsport, Washington"
129,uw714129332,explosion,2.15,-0.62,"15 km S of Princeton, Canada"
185,uw714128982,earthquake,0.22,-2.31,"26 km ENE of Ashford, Washington"


## 8. Tiempos → **Actualidad (Oportunidad)**

**Idea:** `time` (cuándo ocurrió) y `updated` (cuándo USGS publicó *esta versión*) llegan en
**milisegundos desde el 1/1/1970** (Unix epoch). Los convertimos a fecha legible en UTC y calculamos:

- `retraso_min` = `updated − time`: cuánto tardó USGS en dar la versión que vemos.
- `antiguedad_h` = `generated − time`: cuánto hace que ocurrió (debe ser ≤ 24 h, por ser el feed de un día).

**Por qué importa:** si un terremoto se corrige 15 horas después, recibiremos una **versión nueva**
del mismo `id`. Eso es lo que el consumidor tiene que saber gestionar.

In [10]:
# to_datetime con unit="ms" interpreta el número como milisegundos; utc=True lo marca como hora UTC
df["time_utc"]    = pd.to_datetime(df["time"],    unit="ms", utc=True)
df["updated_utc"] = pd.to_datetime(df["updated"], unit="ms", utc=True)
generado = pd.to_datetime(feed["metadata"]["generated"], unit="ms", utc=True)

# Restar dos fechas da una duración; .dt.total_seconds() la pasa a segundos; /60 a minutos, /3600 a horas
df["retraso_min"]  = (df["updated_utc"] - df["time_utc"]).dt.total_seconds() / 60
df["antiguedad_h"] = (generado - df["time_utc"]).dt.total_seconds() / 3600

print("Respuesta generada:", generado)
df[["retraso_min", "antiguedad_h"]].describe().round(2)

Respuesta generada: 2026-10-28 11:36:06+00:00


,retraso_min,antiguedad_h
count,187.00,187.00
mean,87.78,12.84
std,182.37,7.00
min,0.55,0.19
25%,1.63,7.85
50%,3.57,12.83
75%,78.08,19.44
max,925.80,23.99


## 9. Estado de revisión e identificadores asociados

**Idea:**
- `status`: `automatic` = cálculo automático (puede corregirse después → nueva versión) o `reviewed` = revisado por una persona.
- `ids`: texto con los identificadores que distintas redes sísmicas dieron al **mismo** terremoto,
  separados por comas, p. ej. `",us7000ttn0,pr2026301000,"`.

**Por qué importa:** si un terremoto tiene varios ids, USGS los ha asociado y el `id` "preferido" podría
cambiar con el tiempo. No lo resolvemos (es una ampliación), pero hay que **documentarlo como limitación**.

In [11]:
print(df["status"].value_counts(), "\n")

# Contar cuántos ids tiene cada terremoto:
#   .str.strip(",")  quita las comas de los extremos  -> "us7000ttn0,pr2026301000"
#   .str.split(",")  parte el texto por comas          -> ["us7000ttn0", "pr2026301000"]
#   .str.len()       cuenta los elementos de la lista  -> 2
df["n_ids"] = df["ids"].str.strip(",").str.split(",").str.len()

print("Terremotos con más de un id asociado:", (df["n_ids"] > 1).sum())
df.loc[df["n_ids"] > 1, ["id", "ids", "status", "mag"]].head(10)

status
automatic    120
reviewed      67
Name: count, dtype: int64 

Terremotos con más de un id asociado: 8


,id,ids,status,mag
13,pr2026301000,",us7000ttn0,pr2026301000,",reviewed,4.07
74,aka2026vieele,",aka2026vieele,us7000ttl0,",reviewed,3.50
88,hv75064596,",us7000ttkb,hv75064596,",reviewed,2.63
109,us7000ttju,",aka2026vhwvab,us7000ttju,",reviewed,4.20
121,hv75064501,",us7000ttip,hv75064501,",reviewed,2.54
131,aka2026vhpcrg,",aka2026vhpcrg,us7000tti8,",reviewed,3.60
144,aka2026vhmjgz,",aka2026vhmjgz,us7000ttgx,",automatic,2.80
181,tx2026vdtnxx,",us7000ttft,tx2026vdtnxx,",reviewed,2.60


## 10. Conclusiones de la exploración

*(Las redactamos juntos con las cifras que te hayan salido.)*

- 